# f002 head Input接続だけの確認・提出不可

新しいprivate Notebook。Inputsは既存rsna-f002-head-fold0-v1一つだけ、Accelerator None (CPU)、Internet OFF。
Save & Run Allで確認。画像処理・重みロード・学習・推論・提出は行いません。
OutputのF002_HEAD_MOUNT_CHECK.jsonがpassedなら、同じ3ファイルの接続・byte hashまで成功です。
mount dataで停止した場合はPython開始前のためOutput0になり、このJSONも作れません。


In [ ]:
from pathlib import Path
import hashlib, json, traceback, sys

# Attach ONLY this private head Dataset. Accelerator None (CPU), Internet OFF.
# No Competition, encoder, decoder Input, torch, MRI or model inference is used.
HEAD_SLUG = 'rsna-f002-head-fold0-v1'
EXPECTED = {'best.pt': {'bytes': 340745, 'sha256': '189ecab25e0d1c029e974ee23f3c46239e5eceef290b5dca33f6831dcb8e1b03'}, 'config.json': {'bytes': 855, 'sha256': 'f2e98efbb73accd3053032cac811e203d8a4d0570e26e2e32ea9d4eaaaba7280'}, 'checkpoint-provenance.json': {'bytes': 5959, 'sha256': '918231230cfdd21fbae9c766ddf585d17daeef6a6cff9dc14fbf46e2b6ccada6'}}
SUMMARY = Path('/kaggle/working/F002_HEAD_MOUNT_CHECK.json')
if SUMMARY.exists():
    raise RuntimeError('Use a fresh session to preserve the preceding check')
result = {'status': 'started', 'phase': 'python_started', 'python': sys.version.split()[0],
          'head_slug': HEAD_SLUG, 'model_loaded': False, 'training_performed': False,
          'note': 'If Dataset mounting fails before Python, this JSON cannot be created'}
def save():
    SUMMARY.write_text(json.dumps(result, indent=2), encoding='utf-8')
save()
try:
    candidates = [Path('/kaggle/input/datasets/rsraki') / HEAD_SLUG, Path('/kaggle/input') / HEAD_SLUG]
    roots = [candidate for candidate in candidates if candidate.is_dir()]
    if len(roots) != 1:
        raise RuntimeError('Attach exactly one head Input with best.pt/config/provenance at its root')
    root = roots[0]
    result.update(phase='input_accessible', mount_path=str(root), checks={})
    save()
    for name, contract in EXPECTED.items():
        path = root / name
        if not path.is_file() or path.stat().st_size != contract['bytes']:
            raise ValueError('Missing or wrong-size audited head file: ' + name)
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest != contract['sha256']:
            raise ValueError('Audited head content changed: ' + name)
        result['checks'][name] = {'bytes': contract['bytes'], 'sha256': digest, 'matches': True}
        save()
    result.update(status='passed', phase='complete')
    save()
    print(json.dumps(result, indent=2))
except Exception as error:
    result.update(status='failed', error=str(error), traceback=traceback.format_exc())
    save()
    raise
